# TP 3 — Normalisation et transformation des représentations

**Travail réalisé par : Emine OULD AGATT**

Nous étudions deux éléments utilisés dans les blocs Transformer : `LayerNorm`, qui régule les activations d'un token, et le réseau *feed-forward*, chargé de transformer ses caractéristiques sans intervenir sur les autres positions.


In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

## Étape 1 — Programmer une normalisation de couche

In [2]:
class MaLayerNorm:
    def __init__(self, dim, eps=1e-5):
        self.eps = eps
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

    def __call__(self, x):
        moyenne = x.mean(dim=-1, keepdim=True)
        variance = x.var(dim=-1, keepdim=True, unbiased=False)
        x_norm = (x - moyenne) / torch.sqrt(variance + self.eps)
        return self.gamma * x_norm + self.beta

**Questions :** sur quelle dimension appliquer la normalisation ? Quelle fonction remplissent `ε`, `γ` et `β` ?

**Éléments de réponse :**
- La normalisation s'effectue sur le dernier axe (`dim=-1`), c'est-à-dire sur les caractéristiques d'un token donné, indépendamment des autres tokens.
- La petite constante `ε` prévient une division par zéro lorsque la variance est trop faible.
- `γ` et `β` sont des paramètres appris : ils permettent de réajuster respectivement l'échelle et le décalage des activations après normalisation.


## Étape 2 — Vérifier le choix de l'axe normalisé

In [3]:
torch.manual_seed(1337)
mon_module_norm = MaLayerNorm(100)

entrees = torch.randn(32, 100)
entrees_norm = mon_module_norm(entrees)

print("Moyenne/std sur la colonne 0:", entrees_norm[:, 0].mean().item(), entrees_norm[:, 0].std().item())
print("Moyenne/std sur la ligne 0:", entrees_norm[0, :].mean().item(), entrees_norm[0, :].std().item())

Moyenne/std sur la colonne 0: 0.14759676158428192 0.8847486972808838
Moyenne/std sur la ligne 0: -2.2649764730431343e-08 1.005033254623413


**Avant d'exécuter :** déterminer quelle ligne devrait présenter une moyenne proche de 0 et une variance voisine de 1. Confirmer ensuite avec `torch.allclose`.

**Éléments de réponse :**
- La première ligne, d'indice `0`, est normalisée sur ses 100 caractéristiques : sa moyenne doit donc être proche de zéro et sa variance proche de un. Aucune contrainte équivalente n'est imposée sur les colonnes.
- La comparaison `torch.allclose(mon_module_norm(entrees), nn.LayerNorm(100)(entrees), atol=1e-6)` renvoie `True` et confirme que notre réalisation concorde avec celle fournie par PyTorch.


## Étape 3 — Implémenter le réseau feed-forward

In [4]:
class ReseauFeedForward(nn.Module):
    def __init__(self, dim_emb):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim_emb, 4 * dim_emb),
            nn.ReLU(),
            nn.Linear(4 * dim_emb, dim_emb),
        )

    def forward(self, x):
        return self.net(x)

mon_ff = ReseauFeedForward(100)
delta_test = mon_ff(torch.randn(32, 100))
print("Forme à la sortie :", delta_test.shape)

Forme à la sortie : torch.Size([32, 100])


**À expliquer :** quelle part des activations est positive ? Pourquoi la dernière projection doit-elle revenir à `C` caractéristiques ?

**Éléments de réponse :**
- Avec une initialisation approximativement centrée en zéro, près de la moitié des activations sont positives. `ReLU` annule les valeurs négatives, d'où cette proportion indicative.
- L'élargissement temporaire à `4 × C` augmente la capacité de transformation du MLP. Une projection finale vers `C` est indispensable pour additionner sa sortie à l'entrée dans la connexion résiduelle, sans incompatibilité de dimensions.


## Étapes 4 et 5 — Introduire la connexion résiduelle

Nous évaluons ici la variante *pré-norm*, où la normalisation intervient avant le réseau de transformation.

In [5]:
B, T, d_emb = 4, 8, 32

norm = nn.LayerNorm(d_emb)
ff = ReseauFeedForward(d_emb)

x_entree = torch.randn(B, T, d_emb)
delta_ff = ff(norm(x_entree))
x_final = x_entree + delta_ff

print(x_entree.shape)
print(delta_ff.shape)
print(x_final.shape)

torch.Size([4, 8, 32])
torch.Size([4, 8, 32])
torch.Size([4, 8, 32])


**Synthèse — TP 3 :** quelles dimensions sont préservées par `LayerNorm` ? Le MLP combine-t-il plusieurs positions ? Pourquoi ajoute-t-on la sortie de la sous-couche à son entrée ?

**Bilan des observations :**
- Substituer `GELU` à `ReLU` change la forme de l'activation, puisque `GELU` laisse subsister certaines valeurs négatives. Les dimensions des tenseurs restent cependant identiques.
- `LayerNorm` conserve la forme `(B, T, C)` et agit uniquement sur la distribution des caractéristiques.
- Le MLP applique les mêmes transformations à chaque position ; il n'effectue pas d'échange d'information entre les tokens.
- Avec `x = x + delta`, la connexion résiduelle conserve un chemin direct pour le signal et le gradient. Elle facilite ainsi l'optimisation de réseaux profonds, où la disparition des gradients constitue un risque.
